# Week 2 Lab — Stationarity and the Autocorrelation Function
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University

---

### What this lab does

Week 2 said: *stationarity fixes the rules, ergodicity makes one path enough, and the ACF is the fingerprint we read it with.*
This notebook makes that concrete. You will simulate three canonical processes, read their ACFs, and see the
`±2/√n` band behave exactly as the slides claimed.

| Step | What you do |
|---|---|
| 1 | Simulate white noise, AR(1) and a random walk from a common seed |
| 2 | Look at the three paths — which one has "rules that do not change"? |
| 3 | Implement the sample ACF **from the formula**, then check it against `statsmodels` |
| 4 | Overlay the `±2/√n` band and count significant lags |
| 5 | Experiment: `n = 50` vs `n = 1000` |
| 6 | Experiment: `φ = 0.95`, near-nonstationary |
| 7 | Real data — sea surface temperature, ACF before vs after differencing |

### How to use it

Two cells are marked **`TODO`**. Write those yourself first — they are 3–5 lines each and they are the
part worth doing by hand.

> An unfilled `TODO` cell raises `NotImplementedError`. **That is expected, not a broken notebook.**
> Each `TODO` is followed by a collapsed **정답** cell — run it and the notebook continues from there.
> Open it only after you have tried.

> ### ⚠ Before you touch anything: **File > Save a copy in Drive**
> The link I posted opens **read-only**. Colab will say *"changes will not be saved"*.
> Save your own copy first, or everything you type here disappears when you close the tab.
> Work in your copy for the rest of the session.
>
> **Submission format** (from the course notice): share your copy with **Anyone with the link · Viewer**
> and submit that link on the LMS. The last cell prints your environment — run it before you submit.


## 0. Setup

Nothing to install this week — everything below ships with Colab. Later notebooks add an install cell here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

SEED = 42                       # fixed seed = reproducible = required for every submission
rng  = np.random.default_rng(SEED)

plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("numpy      :", np.__version__)
print("statsmodels:", sm.__version__)

---
## 1. Three processes, one seed

The three canonical characters of Week 2:

| Process | Definition | Stationary? |
|---|---|---|
| White noise | $w_t \sim N(0, \sigma^2)$, independent | yes — $\mu=0$, $\gamma(h)=\sigma^2 \mathbb{1}\{h=0\}$ |
| AR(1) | $x_t = \phi x_{t-1} + w_t$, $|\phi|<1$ | yes — $\gamma(h) = \phi^{|h|}\sigma^2/(1-\phi^2)$ |
| Random walk | $x_t = x_{t-1} + w_t$ | **no** — $\gamma(s,t) = \min(s,t)\sigma^2$ |

White noise and the random walk are one line each. AR(1) is yours.

In [ ]:
N = 500

# White noise
wn = rng.normal(0.0, 1.0, size=N)

# Random walk: cumulative sum of white noise (its own innovations)
rw = np.cumsum(rng.normal(0.0, 1.0, size=N))


# ---------------------------------------------------------------- TODO ----
def simulate_ar1(n, phi, sigma=1.0, rng=None):
    """Return one AR(1) path of length n:  x_t = phi * x_{t-1} + w_t,  w_t ~ N(0, sigma^2).

    Start from x_0 = 0. Return a 1-D array of length n.
    """
    rng = np.random.default_rng() if rng is None else rng
    w = rng.normal(0.0, sigma, size=n)
    x = np.zeros(n)
    # TODO: fill in the recursion.
    #       Hint: one loop from t = 1 to n-1, one line inside.
    raise NotImplementedError("simulate_ar1 을 채워 주세요")
# --------------------------------------------------------------------------


ar1 = simulate_ar1(N, phi=0.6, rng=np.random.default_rng(SEED + 1))
print("shapes:", wn.shape, ar1.shape, rw.shape)

In [ ]:
#@title ▶ 정답 — 막혔을 때만 펼쳐서 실행하세요 (여러분 구현을 덮어씁니다) { display-mode: "form" }
def simulate_ar1(n, phi, sigma=1.0, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    w = rng.normal(0.0, sigma, size=n)
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = phi * x[t - 1] + w[t]
    return x

ar1 = simulate_ar1(N, phi=0.6, rng=np.random.default_rng(SEED + 1))
print("ok — ar1 ready, shape", ar1.shape)

---
## 2. Look at the paths first

Before any formula: which of these has *rules that do not change with time*?

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)
for ax, series, name in zip(axes, [wn, ar1, rw],
                            ["White noise", "AR(1), phi = 0.6", "Random walk"]):
    ax.plot(series, lw=0.8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel(name, fontsize=9)
axes[-1].set_xlabel("t")
fig.suptitle("Same seed, three processes", y=0.98)
plt.tight_layout()
plt.show()

print(f"first half  mean/sd | WN {wn[:250].mean():+.3f}/{wn[:250].std():.3f}"
      f"   AR1 {ar1[:250].mean():+.3f}/{ar1[:250].std():.3f}"
      f"   RW {rw[:250].mean():+.3f}/{rw[:250].std():.3f}")
print(f"second half mean/sd | WN {wn[250:].mean():+.3f}/{wn[250:].std():.3f}"
      f"   AR1 {ar1[250:].mean():+.3f}/{ar1[250:].std():.3f}"
      f"   RW {rw[250:].mean():+.3f}/{rw[250:].std():.3f}")

**Read the two lines of output.** For white noise and AR(1) the first and second halves report roughly the
same mean and spread — the rules did not change. For the random walk they do not match at all, and that
mismatch *is* nonstationarity: $\operatorname{Var}(x_t) = t\sigma^2$ keeps growing.

---
## 3. The sample ACF, from the formula

Slide 24 gave it:

$$\hat\gamma(h) = \frac{1}{n}\sum_{t=1}^{n-h}(x_{t+h}-\bar x)(x_t-\bar x), \qquad
\hat\rho(h) = \frac{\hat\gamma(h)}{\hat\gamma(0)}$$

The sum has only $n-h$ terms but we still divide by $n$ — that is what keeps $\hat\gamma$
non-negative definite. Implement it exactly as written.

In [ ]:
# ---------------------------------------------------------------- TODO ----
def sample_acf(x, max_lag=30):
    """Return rho_hat[0..max_lag] using the divide-by-n estimator above.

    rho_hat[0] must be exactly 1.0.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    xbar = x.mean()
    d = x - xbar
    gamma = np.empty(max_lag + 1)
    # TODO: fill gamma[h] for h = 0 .. max_lag, then divide by gamma[0].
    #       Hint: the lag-h product is  d[h:] * d[:n-h]  — sum it and divide by n.
    raise NotImplementedError("sample_acf 를 채워 주세요")
# --------------------------------------------------------------------------


# --- check against statsmodels once you have it ---
mine = sample_acf(ar1, 20)
ref  = sm.tsa.acf(ar1, nlags=20, fft=False)
print("max abs difference vs statsmodels:", np.max(np.abs(mine - ref)))
assert np.allclose(mine, ref, atol=1e-10), "still different — check the divisor"
print("match ✓")

In [ ]:
#@title ▶ 정답 — 막혔을 때만 펼쳐서 실행하세요 { display-mode: "form" }
def sample_acf(x, max_lag=30):
    x = np.asarray(x, dtype=float)
    n = len(x)
    d = x - x.mean()
    gamma = np.array([np.sum(d[h:] * d[:n - h]) / n for h in range(max_lag + 1)])
    return gamma / gamma[0]

mine = sample_acf(ar1, 20)
ref  = sm.tsa.acf(ar1, nlags=20, fft=False)
print("max abs difference vs statsmodels:", np.max(np.abs(mine - ref)))
assert np.allclose(mine, ref, atol=1e-10)
print("match ✓")

---
## 4. Three fingerprints, one band

$\pm 2/\sqrt{n}$ is the 95% band *under white noise*. Bars outside it are worth a second look —
but remember the 1-in-20 rule from slide 26.

In [ ]:
def plot_acf_with_band(ax, x, title, max_lag=30, theo=None):
    n = len(x)
    r = sample_acf(x, max_lag)
    band = 2 / np.sqrt(n)
    ax.bar(range(max_lag + 1), r, width=0.35, color="#3b6ea5")
    ax.axhline(0, color="k", lw=0.7)
    ax.axhspan(-band, band, color="#d95f02", alpha=0.13)
    ax.axhline( band, color="#d95f02", lw=0.9, ls="--")
    ax.axhline(-band, color="#d95f02", lw=0.9, ls="--")
    if theo is not None:
        ax.plot(range(max_lag + 1), theo, "o--", color="#111111",
                ms=3, lw=0.9, label="theoretical")
        ax.legend(fontsize=8)
    n_sig = int(np.sum(np.abs(r[1:]) > band))
    ax.set_title(f"{title}   |  band = ±{band:.3f}  |  {n_sig} of {max_lag} lags outside", fontsize=10)
    ax.set_xlabel("lag h")
    return n_sig

lags = np.arange(31)
fig, axes = plt.subplots(3, 1, figsize=(11, 8))
plot_acf_with_band(axes[0], wn,  "White noise — nothing to model")
plot_acf_with_band(axes[1], ar1, "AR(1), phi = 0.6 — geometric DECAY", theo=0.6 ** lags)
plot_acf_with_band(axes[2], rw,  "Random walk — slow near-linear decay = nonstationarity")
plt.tight_layout()
plt.show()

**What to notice.**

- White noise: a bar or two may poke out. That is the 1-in-20 rule, not structure.
- AR(1): the bars sit on $\rho(h) = \phi^h$ — the theoretical curve is drawn on top. **Decay says AR.**
- Random walk: the slow, near-linear decay is the classic nonstationarity signature.
  Do **not** read it as "long memory to model" — stationarize first, read second.


---
## 5. Experiment A — what `n` does

Slide 25: $n = 100 \to \pm 0.20$, $n = 400 \to \pm 0.10$. Watch two things move at once:
the band tightens, **and** the bars themselves stop scattering.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, n in zip(axes, [50, 200, 1000]):
    x = simulate_ar1(n, phi=0.6, rng=np.random.default_rng(SEED + 7))
    plot_acf_with_band(ax, x, f"AR(1), n = {n}", max_lag=20, theo=0.6 ** np.arange(21))
plt.tight_layout()
plt.show()

print("How far the estimate sits from the truth rho(1) = 0.600:")
for n in [50, 200, 1000]:
    errs = [abs(sample_acf(simulate_ar1(n, 0.6, rng=np.random.default_rng(s)), 1)[1] - 0.6)
            for s in range(300)]
    print(f"  n = {n:5d}   mean |error| = {np.mean(errs):.3f}")

At $n = 50$ the sample ACF is genuinely wild — this is slide 30's *"small samples lie"*, measured.
Anyone reading an ACF off 50 points is mostly reading noise.

---
## 6. Experiment B — $\phi = 0.95$, near the edge

AR(1) is stationary for any $|\phi| < 1$. But at $\phi = 0.95$ it *looks* nonstationary over any
finite sample, and estimating its mean becomes expensive. Slide 21 gave the price:

$$\operatorname{Var}(\bar x) = \frac{1}{n}\sum_{|h|<n}\Big(1 - \frac{|h|}{n}\Big)\gamma(h)
\quad\text{instead of}\quad \frac{\sigma^2}{n}$$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
for ax, phi in zip(axes, [0.6, 0.95]):
    x = simulate_ar1(400, phi=phi, rng=np.random.default_rng(SEED + 11))
    plot_acf_with_band(ax, x, f"AR(1), phi = {phi}", max_lag=40, theo=phi ** np.arange(41))
plt.tight_layout()
plt.show()

# What dependence costs.
# Baseline = gamma(0)/n, i.e. n INDEPENDENT draws with the same marginal variance.
# Any excess is the price of memory alone. Theory says the ratio tends to (1+phi)/(1-phi).
print("Var(xbar) over 2000 replications, n = 400")
print(f"{'phi':>6} {'empirical':>11} {'gamma(0)/n':>12} {'inflation':>10} {'theory':>8}")
for phi in [0.0, 0.6, 0.95]:
    means = [simulate_ar1(400, phi, rng=np.random.default_rng(10_000 + s)).mean()
             for s in range(2000)]
    emp   = np.var(means)
    indep = (1.0 / (1 - phi ** 2)) / 400        # gamma(0)/n
    print(f"{phi:>6.2f} {emp:>11.5f} {indep:>12.5f} {emp/indep:>9.1f}x "
          f"{(1+phi)/(1-phi):>7.1f}x")

The last two columns are the real lesson. The empirical inflation lands on the theoretical
$(1+\phi)/(1-\phi)$, and at $\phi = 0.95$ that is roughly **39×**: 400 correlated observations carry about
the information of ten independent ones. **Effective sample size shrinks with memory** — which is why
slide 23 called long memory "genuinely harder".

---
## 7. Real data — sea surface temperature

`elnino` ships with `statsmodels`: monthly Pacific sea surface temperature, 1950–2010.
Trend plus a strong annual cycle. Watch the ACF before and after differencing.

In [ ]:
raw = sm.datasets.elnino.load_pandas().data
temp = raw.drop(columns="YEAR").to_numpy().ravel()      # 61 years x 12 months, row-major
temp = temp[~np.isnan(temp)]
print("monthly observations:", len(temp))

d1  = np.diff(temp)          # first difference       -> removes the stochastic trend
d12 = temp[12:] - temp[:-12] # seasonal difference    -> removes the annual cycle

fig, axes = plt.subplots(3, 2, figsize=(13, 8))
for row, (series, name) in enumerate([(temp, "Original"),
                                      (d1,   "First difference  (1-B)"),
                                      (d12,  "Seasonal difference  (1-B^12)")]):
    axes[row, 0].plot(series, lw=0.7)
    axes[row, 0].set_title(f"{name} — series", fontsize=10)
    plot_acf_with_band(axes[row, 1], series, f"{name} — ACF", max_lag=40)
plt.tight_layout()
plt.show()

**Read the right-hand column top to bottom.**

- **Original**: bars stay high for many lags, with humps at 12, 24, 36. Slide 27's reading key calls this
  *persistence or nonstationarity* plus *seasonality*.
- **First difference**: the slow decay collapses, but the lag-12 spikes survive — differencing killed
  the trend, not the annual cycle.
- **Seasonal difference**: the 12-lag structure goes too.

That is the whole diagnostic loop: *difference, re-read the ACF, repeat until the bars fall into the band.*
Week 7 turns "until the bars fall in" into a formal unit-root test.

---
## 8. Your turn

Small changes, real answers. Do at least two.

1. Set `phi = -0.6`. What happens to the ACF, and why? (Look at the **sign** pattern.)
2. Simulate MA(1), $x_t = w_t + \theta w_{t-1}$ with $\theta = 0.8$. Confirm the ACF **cuts off** after
   lag 1 rather than decaying — this is the fingerprint that separates MA from AR.
3. Take the random walk and difference it once. What should the ACF look like, and does it?
4. Re-run section 7 with `d12` differenced **again** by 1. Did anything improve, or did you over-difference?

Write one or two sentences under each thing you tried. The sentences are the point, not the plots.

In [ ]:
# Scratch space for section 8.


---
## 9. Environment record — run this before you submit

Every submission in this course carries this cell. It is how a marker reproduces your numbers.

In [ ]:
import sys, platform, datetime
try:
    import torch; gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
except Exception:
    gpu = "torch not loaded"

print("run at      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("python      :", sys.version.split()[0], "on", platform.system())
print("runtime     :", gpu)
print("seed        :", SEED)
print("numpy       :", np.__version__)
print("statsmodels :", sm.__version__)